# Diagnóstico do AutoGluon no Brazilian Houses

**Pergunta:** por que o AutoGluon ficou muito pior que as outras ferramentas nesse dataset (RMSE médio ≈ 9.500 contra 1.200 a 4.400), com superestimativas enormes em poucos pontos, mesmo sem o vazamento por duplicatas?

**Hipóteses que este notebook separa:**
- **H1 – empilhamento (stacking):** os modelos de nível 2 e 3 amplificam o valor extremo (R$ 1.120.000) visto no treino.
- **H2 – redes neurais / KNN:** algum desses modelos extrapola para ~R$ 1 milhão em imóveis comuns.
- **H3 – escala do alvo:** a cauda longa do alvo; transformar com `log1p` resolveria.

**Como usar (sem subir arquivo nenhum):**
1. Runtime com T4 (a mesma do estudo) ou CPU, tanto faz para o AutoGluon.
2. Rode as células em ordem. O **Estágio 1** (~8 min) já mostra *qual modelo* gera os erros. O **Estágio 2** (~45 min) roda as variantes que testam as hipóteses; reduza `VARIANTES_RODAR` ou `SEEDS` se quiser economizar tempo.
3. Tudo é salvo em `Meu Drive/TCC_resultados/diagnostico_autogluon/` e o notebook é **retomável** (pula o que já foi feito).
4. No fim, me mande a tabela impressa na última célula (ou o arquivo `resumo_variantes.csv`).

**Atenção:** a variante `sem_empilhamento` desliga o *dynamic stacking*, que gasta cerca de 150 s do orçamento; ela acaba tendo mais tempo efetivo para treinar. Anote isso na comparação.

In [ ]:
# 1) Instalação (mesmas versões do estudo)
!pip install -q --default-timeout=200 autogluon.tabular==1.5.0 catboost==1.2.10
# Se o Colab pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e rode daqui para baixo.

In [ ]:
# 2) Drive, pastas e registro do ambiente
import os, sys, json, time, shutil, warnings, platform
from importlib import metadata
from google.colab import drive
drive.mount("/content/drive")
DIR = "/content/drive/MyDrive/TCC_resultados"          # resultados do estudo (já existem aqui)
OUT = f"{DIR}/diagnostico_autogluon"
os.makedirs(OUT, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
try:
    import torch
    cuda = bool(torch.cuda.is_available())
except Exception:
    cuda = None
amb = {"python": platform.python_version(), "cuda_disponivel": cuda,
       "autogluon.tabular": metadata.version("autogluon.tabular"), "catboost": metadata.version("catboost"),
       "scikit-learn": metadata.version("scikit-learn"), "pandas": pd.__version__, "numpy": np.__version__}
json.dump(amb, open(f"{OUT}/ambiente_diagnostico.json", "w"), indent=2)
print(amb)
print("Resultados do estudo encontrados no Drive:", len([f for f in os.listdir(DIR) if f.startswith("autogluon__brazilian")]), "arquivos do AutoGluon/Brazilian")

In [ ]:
# 3) Dados e partições (idênticos ao estudo: train_test_split 80/20 com random_state = seed)
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

d = fetch_openml(data_id=42688, as_frame=True, parser="auto")      # Brazilian Houses (OpenML)
X, y = d.data, pd.to_numeric(d.target, errors="coerce")
print("Dados:", X.shape, "| categóricas:", list(X.select_dtypes(exclude="number").columns))
print("Maiores alvos:", y.sort_values(ascending=False).head(6).round(0).tolist(), "| mediana:", float(y.median()))

dfh = X.copy(); dfh["__y__"] = y.values
H = pd.util.hash_pandas_object(dfh, index=False)                    # identifica linhas idênticas (X + alvo)
print("Linhas duplicadas (X + alvo):", int(H.duplicated().sum()))

def particao(seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=seed)
    vazou = H.loc[Xte.index].isin(set(H.loc[Xtr.index])).values      # linha do teste idêntica a uma do treino
    return Xtr, Xte, ytr, yte, vazou

def conferir_com_estudo(seed):
    # compara a partição de agora com a do estudo (se o arquivo de previsões estiver no Drive)
    f = f"{DIR}/autogluon__brazilian_houses__seed{seed}_pred.npz"
    if not os.path.exists(f):
        return print(f"seed {seed}: sem previsões do estudo no Drive para conferir")
    z = np.load(f); _, _, _, yte, _ = particao(seed)
    print(f"seed {seed}: partição idêntica à do estudo? {np.allclose(z['y'], yte.values)}")

for s in range(42, 47):
    _, _, ytr, yte, vz = particao(s)
    print(f"seed {s}: maior alvo no teste = {yte.max():,.0f} | no treino = {ytr.max():,.0f} | linhas do teste duplicadas do treino = {int(vz.sum())}")
for s in (44, 45):
    conferir_com_estudo(s)

In [ ]:
# 4) Funções do diagnóstico
TIME_LIMIT = 300      # igual ao estudo

VARIANTES = {
    "base":             dict(fit=dict(presets="high_quality"), log=False),                      # configuração do estudo
    "sem_empilhamento": dict(fit=dict(presets="high_quality", num_stack_levels=0, dynamic_stacking=False), log=False),   # H1
    "sem_nn_knn":       dict(fit=dict(presets="high_quality", excluded_model_types=["NN_TORCH", "FASTAI", "KNN"]), log=False),  # H2
    "log1p":            dict(fit=dict(presets="high_quality"), log=True),                       # H3 (alvo em log1p, previsão com expm1)
}

def metricas(yv, pv, vazou):
    e = pv - yv
    ok = ~vazou
    return dict(rmse=float(np.sqrt(np.mean(e ** 2))), mae=float(np.mean(np.abs(e))),
                rmse_sem_dup=float(np.sqrt(np.mean(e[ok] ** 2))), mae_sem_dup=float(np.mean(np.abs(e[ok]))),
                maior_superestimativa=float(e.max()), maior_subestimativa=float((-e).max()))

def achar_pesos(obj, caminho=""):
    # procura recursivamente 'model_weights' (pesos do WeightedEnsemble) no dicionário info() do AutoGluon
    achados = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "model_weights" and isinstance(v, dict):
                achados[caminho or "raiz"] = v
            else:
                achados.update(achar_pesos(v, f"{caminho}/{k}" if caminho else str(k)))
    return achados

def rodar(nome, seed, time_limit=TIME_LIMIT):
    from autogluon.tabular import TabularPredictor
    cfg = VARIANTES[nome]
    Xtr, Xte, ytr, yte, vazou = particao(seed)
    f = (lambda v: np.log1p(v)) if cfg["log"] else (lambda v: v)
    finv = (lambda v: np.expm1(v)) if cfg["log"] else (lambda v: v)
    tr = Xtr.copy(); tr["__y__"] = f(ytr.values)
    path = f"/content/ag_diag/{nome}_{seed}"
    shutil.rmtree(path, ignore_errors=True)
    t0 = time.time()
    p = TabularPredictor(label="__y__", problem_type="regression", eval_metric="root_mean_squared_error", path=path, verbosity=1)
    p.fit(tr, time_limit=time_limit, ag_args_fit={"random_seed": seed}, **cfg["fit"])
    tempo = time.time() - t0
    yv = yte.values
    melhor = p.model_best
    pred_final = finv(p.predict(Xte).values)
    reg = dict(variante=nome, seed=seed, tempo_s=round(tempo, 1), modelo_final=melhor, **metricas(yv, pred_final, vazou))

    # previsão de CADA modelo (é aqui que aparece quem gera as superestimativas)
    try:
        preds = p.predict_multi(Xte)
    except Exception as e:
        print("predict_multi indisponível, usando só o modelo final:", repr(e)[:150])
        preds = {melhor: p.predict(Xte)}
    lb = p.leaderboard(silent=True).set_index("model")
    lb.reset_index().to_csv(f"{OUT}/{nome}__seed{seed}__leaderboard.csv", index=False)   # lista TODOS os modelos (inclusive sem _FULL, com score_val)
    linhas = []
    for m, pv in preds.items():
        row = dict(modelo=m, **metricas(yv, finv(np.asarray(pv)), vazou))
        for c in ("stack_level", "score_val", "fit_time"):
            if m in lb.index and c in lb.columns:
                row[c] = lb.loc[m, c]
        linhas.append(row)
    por_modelo = pd.DataFrame(linhas).sort_values("rmse").reset_index(drop=True)

    # maiores erros do modelo final, com as características do imóvel
    err = pred_final - yv
    idx = np.argsort(-np.abs(err))[:8]
    top = Xte.iloc[idx].copy()
    top.insert(0, "y_real", yv[idx]); top.insert(1, "previsto", pred_final[idx].round(0))
    top.insert(2, "erro", err[idx].round(0)); top.insert(3, "copia_no_treino", vazou[idx])

    try:
        pesos = achar_pesos(p.info()["model_info"].get(melhor, {}))
    except Exception as e:
        pesos = {"erro": repr(e)[:150]}
    shutil.rmtree(path, ignore_errors=True)
    base = f"{OUT}/{nome}__seed{seed}"
    por_modelo.to_csv(base + "__por_modelo.csv", index=False)
    top.to_csv(base + "__maiores_erros.csv")
    json.dump({"resumo": reg, "pesos": pesos}, open(base + "__resumo.json", "w"), indent=2, default=str)
    return reg, por_modelo, top, pesos

def comparar_com_estudo(seed, rmse_novo):
    f = f"{DIR}/autogluon__brazilian_houses__seed{seed}.json"
    if os.path.exists(f):
        r = json.load(open(f, encoding="utf-8"))["rmse"]
        print(f"RMSE do estudo (seed {seed}) = {r:,.0f}  |  RMSE agora (base) = {rmse_novo:,.0f}  -> reproduz? {abs(r - rmse_novo) / r < 0.15}")

def mostrar(reg, por_modelo, top, pesos):
    from IPython.display import display
    pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
    print(f"=== {reg['variante']} | seed {reg['seed']} | modelo final: {reg['modelo_final']} | tempo {reg['tempo_s']} s ===")
    print(f"RMSE = {reg['rmse']:,.0f} | MAE = {reg['mae']:,.0f} | RMSE sem duplicatas = {reg['rmse_sem_dup']:,.0f} | maior superestimativa = {reg['maior_superestimativa']:,.0f}")
    cols = [c for c in ("modelo", "stack_level", "rmse", "mae", "maior_superestimativa", "maior_subestimativa") if c in por_modelo.columns]
    print("\n-- Os 8 modelos com MENOR RMSE no teste --");  display(por_modelo[cols].head(8).round(0))
    print("-- Os 8 modelos com MAIOR RMSE no teste (suspeitos) --"); display(por_modelo[cols].tail(8).round(0))
    print("-- Pesos do ensemble final --")
    for caminho, pesos_ in pesos.items():
        if isinstance(pesos_, dict):
            print({k: round(v, 3) for k, v in sorted(pesos_.items(), key=lambda kv: -kv[1])[:10]})
        else:
            print(pesos_)
    print("\n-- 8 maiores erros do modelo final --"); display(top)
print("Funções prontas. Variantes:", list(VARIANTES))

## Estágio 1 — quem gera as superestimativas? (~8 min)
Roda a configuração **idêntica à do estudo** na seed 44 (onde o AutoGluon teve RMSE ≈ 17.000) e abre o modelo por modelo.
Também confere se o resultado **reproduz** o do estudo (se não reproduzir, a diferença já é uma informação sobre a variabilidade do AutoGluon).

In [ ]:
# ESTÁGIO 1
reg, por_modelo, top, pesos = rodar("base", 44)
mostrar(reg, por_modelo, top, pesos)
comparar_com_estudo(44, reg["rmse"])

**Como ler o Estágio 1:**
- Se os modelos com **maior RMSE** forem `NeuralNet...` ou `KNeighbors...` → aponta para **H2** (a variante `sem_nn_knn` deve corrigir).
- Se forem modelos de **nível 2 ou 3** (`..._L2`, `..._L3`, `WeightedEnsemble_L3`) enquanto os de nível 1 estão bem → aponta para **H1** (`sem_empilhamento`).
- Se *todos* os modelos tiverem grandes superestimativas nos mesmos imóveis → aponta para **H3** (cauda do alvo; `log1p`).
- "copia_no_treino = True" na tabela de erros indica linha duplicada do treino.

## Estágio 2 — variantes que testam as hipóteses (~35 min)
Roda cada variante nas seeds escolhidas, com o mesmo orçamento (300 s). É retomável: se o Colab cair, rode a célula de novo.

In [ ]:
# ESTÁGIO 2
SEEDS = [44, 45]
VARIANTES_RODAR = ["base", "sem_nn_knn", "log1p"]   # "sem_empilhamento" fica de fora: no Estágio 1 o ensemble final só tinha 1 nível de empilhamento

resultados = []
for seed in SEEDS:
    for nome in VARIANTES_RODAR:
        arq = f"{OUT}/{nome}__seed{seed}__resumo.json"
        if os.path.exists(arq):
            resultados.append(json.load(open(arq))["resumo"]); print("já feito:", nome, seed); continue
        try:
            r, pm, tp, ps = rodar(nome, seed)
            resultados.append(r)
            print(f"OK {nome} seed{seed}: RMSE={r['rmse']:,.0f} MAE={r['mae']:,.0f} RMSE sem dup={r['rmse_sem_dup']:,.0f} tempo={r['tempo_s']}s")
        except Exception as e:
            print("FALHOU", nome, seed, "->", repr(e)[:400])

In [ ]:
# RESULTADO FINAL: tabela para me enviar
from IPython.display import display
resumo = pd.DataFrame(resultados)
def rmse_estudo(seed):
    f = f"{DIR}/autogluon__brazilian_houses__seed{seed}.json"
    return json.load(open(f, encoding="utf-8"))["rmse"] if os.path.exists(f) else np.nan
resumo["rmse_estudo_mesma_seed"] = resumo["seed"].map(rmse_estudo)
cols = ["variante", "seed", "rmse", "mae", "rmse_sem_dup", "mae_sem_dup", "maior_superestimativa", "tempo_s", "modelo_final", "rmse_estudo_mesma_seed"]
resumo = resumo[cols].sort_values(["seed", "variante"])
resumo.to_csv(f"{OUT}/resumo_variantes.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(resumo.round(0))
print("\nMédia entre seeds por variante:")
display(resumo.groupby("variante")[["rmse", "mae", "rmse_sem_dup", "maior_superestimativa"]].mean().round(0))
shutil.make_archive("/content/diagnostico_autogluon", "zip", OUT)
print("\nArquivos salvos em:", OUT, "(zip em /content/diagnostico_autogluon.zip)")